# Data and loss

Training data in idris-ml is typed and indexed. A `Dataset` knows its size and how to
materialise the sample at each **in-bounds** index; samples are `(input, target)` tensor
pairs whose dimensions are in the type, so you can't feed a 2-feature input to a model that
expects 4.

## Dataset

`Dataset` is one of PyTorch's three orthogonal joints (indexed access); ordering and
batching live in `DataStream`. `item` takes a `Fin size`, so out-of-bounds access is
**unrepresentable** — no runtime bounds check, no partiality:

```idris
record Dataset (sample : Type) where
  constructor MkDataset
  size : Nat
  item : Fin size -> IO sample
```

In [ ]:
:doc Dataset

In [ ]:
:t fromVect

## `Fin`: bounds in the type

`Fin n` is the type of naturals `< n`. Indexing with it can never go out of bounds — and
an out-of-range literal is a *compile* error. The next cell is *expected to fail* (`5` is
not a valid `Fin 3`):

In [ ]:
:exec printLn (the (Fin 3) 5)

## A sample is a tensor pair

For a 2-feature, 3-class point, the input is `Tensor [2]` and the one-hot target is
`Tensor [3]`. `batched` later collates samples into `([b,2], [b,3])` batches C-side:

In [ ]:
:exec do {
  x <- tensor {dims=[2]} {ex=TapeExecutor} {dt=F64} (FromVect [1.5, -2.7]);
  y <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 1.0, 0.0]);
  putStrLn ("input[0] = " ++ show (primItem1d {ex=TapeExecutor} x.tensorPtr 0)
            ++ ", target argmax class = 1") }

## Loss functions

Tensor-level losses (IO-typed, `IsFloating dt =>`). For classification, apply
`tlogSoftmax1d` to raw logits and feed `tnllLoss` — do **not** put a softmax layer in the
model (it creates `1/p` intermediates that blow up).

In [ ]:
:t tnllLoss

In [ ]:
:t tmseLoss

In [ ]:
:t tbceLoss

They map to PyTorch:
- `tlogSoftmax1d` + `tnllLoss` = `nn.CrossEntropyLoss` (apply to logits)
- `tnllLoss` = `nn.NLLLoss` (expects log-probabilities)
- `tmseLoss` = `nn.MSELoss` — note this is a **sum** reduction; scale by `1/n` for PyTorch's mean default
- `tbceLoss` = `nn.BCELoss`

## Computing a loss

Forward pass produces an output; the loss measures how wrong it is. Here, MSE between a prediction and a one-hot target:

In [ ]:
:exec do {
  p <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.1, 0.9, 0.0]);
  t <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 1.0, 0.0]);
  l <- tmseLoss (retypeGrad p) (retypeGrad t);
  putStrLn ("MSE (sum) = " ++ show (tensorItem l)) }

This is the building block of training: forward → loss → backward → step. The next notebook ([04 Training](04_training.ipynb)) wires it to an optimizer via `fit`.